# Erlang Loss Model

In [ ]:
import pytangram as tg

In [ ]:
# --- Set parameter values here ---
LAMBDA  = 2.0
MU      = 1.0
SIGMA1  = 1.0
SIGMA2  = 1.0

In [ ]:
class Model(tg.Object):
    n1 = tg.var(0, max=100)
    n2 = tg.var(0, max=100)

    lam = LAMBDA
    mu  = MU
    sigma1 = SIGMA1
    sigma2 = SIGMA2

    @tg.event(dist=tg.EXP("lam"))
    def ev_n1_inc(self, state):
        if state["n1"] >= 100:
            return tg.DISABLED
        return state.set(n1=state["n1"] + 1)

    @tg.event(dist=tg.EXP("mu"))
    def ev_n1_dec(self, state):
        if state["n1"] <= 0:
            return tg.DISABLED
        return state.set(n1=state["n1"] - 1)

    @tg.event(dist=tg.EXP(lambda s: s["n1"] * SIGMA1))
    def ev_n1_dec_n2_inc(self, state):
        if state["n1"] <= 0:
            return tg.DISABLED
        return state.set(n1=state["n1"] - 1, n2=state["n2"] + 1)

    @tg.event(dist=tg.EXP(lambda s: s["n2"] * SIGMA2 + (MU if s["n1"] == 0 and MU > 0 else 0)))
    def ev_n2_dec(self, state):
        if state["n2"] <= 0:
            return tg.DISABLED
        return state.set(n2=state["n2"] - 1)

    @tg.impulse_reward(on_event="ev_n1_dec")
    def count_mu(self, state):
        return 1.0

    @tg.impulse_reward(on_event="ev_n1_dec_n2_inc")
    def count_sigma1(self, state):
        return 1.0

    @tg.impulse_reward(on_event="ev_n2_dec")
    def count_n2_dec(self, state):
        return 1.0

In [ ]:
model = tg.Model(Model(), name="ErlangLoss")
sol = model.solve()

print(f"States:       {sol.space.n_states}")
print(f"Transitions:  {sol.space.n_transitions}")
print(sol.all_rewards())